# Lab 2 — Synthetic memory bootstrap

Goal: provision useful operating context before an agent has real history. Synthetic memory is not ground truth about a user; it is a controlled way to seed workflows, entities, policies, skills, and tool descriptions for a demo or new domain.


## 0. Generate the bundle offline

The helper is deterministic by default. You can run this cell without Oracle or an LLM provider.


In [ ]:
from pathlib import Path
import json

from agentic_memory_workshop.synthesize_memory import generate_bundle, write_bundle

bundle = generate_bundle(
    user_id='workshop_user',
    agent_id='memory_agent',
    domain='restaurant concierge',
    thread_count=20,
    workflow_count=15,
    entity_count=30,
    knowledge_count=50,
    skill_count=5,
    tool_count=10,
    seed=7,
)
out = Path('../generated/synthetic_memory.json')
write_bundle(bundle, out, write_skill_files=True)
print(f'Wrote {out}')


## 1. Inspect what was generated

The bundle covers each memory category we want to seed: conversational/episodic threads, workflows, entities, semantic knowledge, skills, and tools.


In [ ]:
data = json.loads(out.read_text())
for key in ['threads', 'workflows', 'entities', 'knowledge', 'skills', 'tools']:
    print(f'{key:10s}', len(data[key]))

print('\nExample entity memory:')
print(data['entities'][0]['content'])
print('\nExample skill:')
print(data['skills'][0]['body'])


## 2. Load synthetic memory into OAMP

This loader uses the same API surface as Lab 1: `add_user`, `add_agent`, `create_thread`, `thread.add_messages`, and `add_memory`. With `OAMP_EMBEDDER=oracle-db`, embeddings run inside Oracle.


In [ ]:
from agentic_memory_workshop.connection import load_env
from agentic_memory_workshop.synthesize_memory import load_into_oamp

load_env('../.env')
loaded = load_into_oamp(bundle)
loaded


## 3. Prove first-turn retrieval

Now create a fresh client and ask a question that should retrieve synthetic context even though this thread has no real conversation history.


In [ ]:
from agentic_memory_workshop.connection import connect
from agentic_memory_workshop.onnx import oamp_embedder
from oracleagentmemory.core import OracleAgentMemory

conn = connect()
client = OracleAgentMemory(
    connection=conn,
    embedder=oamp_embedder(conn),
    extract_memories=False,
    schema_policy='create_if_necessary',
)
results = client.search('How should the agent handle a refund request for a restaurant booking?', user_id='workshop_user', max_results=5)
for result in results:
    print(f'[{result.record.record_type}] distance={result.distance:.3f}')
    print(result.content[:300])
    print('---')


## 4. Optional: load synthetic tools and skills

Run this after the in-database ONNX embedding model is available. It seeds the same registries used in the mini-lab.


In [ ]:
from agentic_memory_workshop.synthesize_memory import load_toolbox_skillbox

try:
    load_toolbox_skillbox(bundle)
except Exception as exc:
    print('Toolbox/skillbox load skipped or failed. Most common cause: ONNX model not loaded.')
    print(type(exc).__name__, exc)


## 5. Compare against naive flat history

In the live session, the instructor shows the benchmark chart from the OAMP benchmarks notebook. Your checkpoint for Packt completion is the generated bundle plus a screenshot of retrieval results from step 3.

Discussion prompts:

- Which generated records are safe bootstrap context?
- Which records should never be synthetic without review?
- What governance rule would you add before loading synthetic memories in production?
